# Microcosmic God → Catch Transfer Test

This notebook tests whether brains evolved in the [microcosmic-god](https://github.com/asystemoffields/microcosmic-god) artificial-life sandbox have transferable representations — does the cognition that emerged in a complex causal-survival world transfer to a simple paddle-and-ball game it has never seen?

## The protocol

1. Load a brain from a microcosmic-god checkpoint. **Brain weights stay frozen — no further learning.**
2. Train a small linear adapter that maps Catch's 4-dim observation into the brain's 72-dim input space, and the brain's 15-dim output into Catch's 3 actions.
3. Compare three conditions:
   - **Random policy** — left/stay/right uniformly. Lower bound.
   - **Untrained brain + trained adapter** — brain weights random-init; only adapter learns. Controls for what the adapter alone can do.
   - **Trained brain (loaded checkpoint) + trained adapter** — the actual transfer test.

If condition 3 outperforms condition 2 meaningfully, the brain's internal representations are doing useful work in a task they were never trained on. That's a real signal of transferability.

**Why a linear adapter and not a deeper one?** Deep adapters can learn to compute Catch from scratch, hiding any (lack of) contribution from the brain. A small linear projection is the cleanest test: the brain has to be doing something useful for the adapter to extract a working policy.

## Setup

Pure numpy + matplotlib. Clones the microcosmic-god repo to access sample checkpoints.

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

if not os.path.exists('microcosmic-god'):
    !git clone -q https://github.com/asystemoffields/microcosmic-god.git

BRAIN_REPO = 'microcosmic-god' if os.path.exists('microcosmic-god') else '.'
print('Using brain repo at:', BRAIN_REPO)

## The Catch environment

10×10 grid. Paddle on bottom row, 3 cells wide. Ball drops one row per tick from a random column. Three actions: left / stay / right. Reward +1 if caught, −1 if missed. Each episode is exactly 9 steps (the ball reaches the paddle row).

Observation: `[paddle_x, ball_x, ball_y, t]` all normalized to [0, 1] — 4 floats.

In [ ]:
class Catch:
    def __init__(self, size=10, paddle_width=3, rng=None):
        self.size = size
        self.paddle_width = paddle_width
        self.rng = rng if rng is not None else np.random.default_rng()
        self.reset()

    def reset(self):
        self.paddle_x = self.size // 2
        self.ball_x = int(self.rng.integers(0, self.size))
        self.ball_y = 0
        self.t = 0
        return self._obs()

    def _obs(self):
        return np.array([
            self.paddle_x / max(1, self.size - 1),
            self.ball_x / max(1, self.size - 1),
            self.ball_y / max(1, self.size - 1),
            self.t / max(1, self.size - 1),
        ], dtype=np.float64)

    def step(self, action: int):
        if action == 0:
            self.paddle_x = max(0, self.paddle_x - 1)
        elif action == 2:
            self.paddle_x = min(self.size - 1, self.paddle_x + 1)
        self.ball_y += 1
        self.t += 1
        done = self.ball_y >= self.size - 1
        reward = 0.0
        if done:
            half = self.paddle_width // 2
            distance = abs(self.ball_x - self.paddle_x)
            reward = 1.0 if distance <= half else -1.0
        return self._obs(), reward, done

## Brain inference

Loads a microcosmic-god `TinyBrain` checkpoint. Pure numpy, forward-pass only — no plasticity. Mirrors the math in `microcosmic_god/brain.py`'s `forward()` and `_attend()`. Attention noise is set to zero for deterministic evaluation (we're testing what the brain *learned*, not its noise tolerance).

In [ ]:
class TinyBrainInference:
    def __init__(self, weights_in, weights_out, bias_h, bias_o,
                 attention_weights=None, attention_bias=None):
        self.weights_in = weights_in
        self.weights_out = weights_out
        self.bias_h = bias_h
        self.bias_o = bias_o
        self.attention_weights = attention_weights if (attention_weights is not None and attention_weights.size) else None
        self.attention_bias = attention_bias if (attention_bias is not None and attention_bias.size) else None
        self.input_size = weights_in.shape[1]
        self.hidden_size = weights_in.shape[0]
        self.output_size = weights_out.shape[0]
        self.hidden = np.zeros(self.hidden_size)

    @classmethod
    def from_checkpoint(cls, path):
        with open(path) as f:
            data = json.load(f)
        # Microcosmic-god checkpoints wrap the brain in 'brain' or 'brain_template'.
        brain_data = data.get('brain') or data.get('brain_template') or data
        h = int(brain_data['hidden_size'])
        i = int(brain_data['input_size'])
        o = int(brain_data['output_size'])
        weights_in = np.array(brain_data['weights_in'], dtype=np.float64).reshape(h, i)
        weights_out = np.array(brain_data['weights_out'], dtype=np.float64).reshape(o, h)
        bias_h = np.array(brain_data['bias_h'], dtype=np.float64)
        bias_o = np.array(brain_data['bias_o'], dtype=np.float64)
        aw = brain_data.get('attention_weights') or []
        ab = brain_data.get('attention_bias') or []
        attention_weights = np.array(aw, dtype=np.float64).reshape(h, i) if len(aw) == h * i else None
        attention_bias = np.array(ab, dtype=np.float64) if len(ab) == i else None
        return cls(weights_in, weights_out, bias_h, bias_o, attention_weights, attention_bias)

    @classmethod
    def random(cls, input_size, hidden_size, output_size, rng=None, with_attention=True):
        rng = rng or np.random.default_rng(0)
        weights_in = rng.normal(0, 0.45, (hidden_size, input_size))
        weights_out = rng.normal(0, 0.45, (output_size, hidden_size))
        bias_h = rng.normal(0, 0.08, hidden_size)
        bias_o = rng.normal(0, 0.08, output_size)
        if with_attention:
            attention_weights = rng.normal(0, 0.15, (hidden_size, input_size))
            attention_bias = rng.normal(3.0, 0.10, input_size)
        else:
            attention_weights = None
            attention_bias = None
        return cls(weights_in, weights_out, bias_h, bias_o, attention_weights, attention_bias)

    def reset(self):
        self.hidden = np.zeros(self.hidden_size)

    def _attend(self, x, budget_fraction=0.95):
        if self.attention_weights is None:
            return x
        inv_h = 1.0 / np.sqrt(max(1, self.hidden_size))
        logits = self.attention_bias + (self.hidden @ self.attention_weights) * inv_h
        np.clip(logits, -30.0, 30.0, out=logits)
        raw = 1.0 / (1.0 + np.exp(-logits))
        budget = self.input_size * budget_fraction
        total = raw.sum()
        fidelity = raw * (budget / total) if total > budget and total > 0 else raw
        return x * fidelity  # noise=0 in transfer mode

    def forward(self, inputs):
        x = self._attend(np.asarray(inputs, dtype=np.float64))
        inv = 1.0 / np.sqrt(max(1, self.input_size))
        new_hidden = np.tanh(self.bias_h + 0.62 * self.hidden + (self.weights_in @ x) * inv)
        self.hidden = new_hidden
        inv_h = 1.0 / np.sqrt(max(1, self.hidden_size))
        return self.bias_o + (self.weights_out @ self.hidden) * inv_h

## The adapter

Two trainable linear projections wrapped around the (frozen) brain:
- `W_in` — Catch's 4-dim observation → brain's input space (72 dims by default)
- `W_out` — brain's 15-dim output → 3 Catch actions

The brain itself is unchanged. If the brain's hidden representations are useful for Catch, training only these projections should produce a competent player.

In [ ]:
class CatchAdapter:
    def __init__(self, brain, catch_obs_size=4, catch_action_size=3, rng=None):
        self.brain = brain
        self.catch_obs_size = catch_obs_size
        self.catch_action_size = catch_action_size
        self.rng = rng or np.random.default_rng()
        self.W_in = self.rng.normal(0, 0.30, (brain.input_size, catch_obs_size))
        self.b_in = np.zeros(brain.input_size)
        self.W_out = self.rng.normal(0, 0.30, (catch_action_size, brain.output_size))
        self.b_out = np.zeros(catch_action_size)

    def reset(self):
        self.brain.reset()

    def policy_logits(self, catch_obs):
        brain_in = self.W_in @ catch_obs + self.b_in
        brain_out = self.brain.forward(brain_in)
        return self.W_out @ brain_out + self.b_out

    def act_greedy(self, catch_obs):
        return int(np.argmax(self.policy_logits(catch_obs)))

    def act_stochastic(self, catch_obs, temperature=1.0):
        logits = self.policy_logits(catch_obs) / max(1e-6, temperature)
        probs = np.exp(logits - logits.max())
        probs /= probs.sum()
        return int(self.rng.choice(len(probs), p=probs))

    def get_params(self):
        return self.W_in.copy(), self.W_out.copy(), self.b_in.copy(), self.b_out.copy()

    def set_params(self, W_in, W_out, b_in, b_out):
        self.W_in = W_in
        self.W_out = W_out
        self.b_in = b_in
        self.b_out = b_out

## Training (Evolution Strategies)

We can't backprop through the brain in pure numpy without a deep-learning library, and we don't want to — the test is purest if the brain is genuinely frozen with no gradient flow. Instead we use Evolution Strategies (ES): perturb the adapter weights with small gaussian noise, evaluate, and update toward perturbations that improved performance. Slow but framework-free, and keeps the brain truly frozen.

In [ ]:
def run_episode(adapter, env, greedy=False):
    adapter.reset()
    obs = env.reset()
    total = 0.0
    done = False
    while not done:
        action = adapter.act_greedy(obs) if greedy else adapter.act_stochastic(obs, temperature=1.0)
        obs, reward, done = env.step(action)
        total += reward
    return total

def evaluate(adapter, env, n_episodes=200):
    return float(np.mean([run_episode(adapter, env, greedy=True) for _ in range(n_episodes)]))

def train_es(adapter, env, generations=150, pop=32, sigma=0.20, lr=0.20,
             n_avg=3, log_every=10, seed=0, verbose=True):
    """Evolution Strategies. n_avg episodes are averaged per perturbation
    to reduce variance from Catch's stochastic ball position - without
    this the gradient estimate is too noisy and learning thrashes."""
    base = adapter.get_params()
    history = []
    rng = np.random.default_rng(seed)
    for gen in range(generations):
        perturbations = []
        rewards = []
        for _ in range(pop):
            eps = tuple(rng.normal(0, sigma, p.shape) for p in base)
            adapter.set_params(*[b + e for b, e in zip(base, eps)])
            r = float(np.mean([run_episode(adapter, env, greedy=False) for _ in range(n_avg)]))
            perturbations.append(eps)
            rewards.append(r)
        rewards = np.array(rewards)
        if rewards.std() > 1e-8:
            normed = (rewards - rewards.mean()) / rewards.std()
        else:
            normed = rewards - rewards.mean()
        deltas = [np.zeros_like(p) for p in base]
        for r, eps in zip(normed, perturbations):
            for i in range(len(deltas)):
                deltas[i] += r * eps[i]
        base = tuple(b + (lr / (pop * sigma)) * d for b, d in zip(base, deltas))
        adapter.set_params(*base)
        if gen % log_every == 0 or gen == generations - 1:
            avg = evaluate(adapter, env, n_episodes=40)
            history.append((gen, float(avg)))
            if verbose:
                print(f'  gen {gen:3d}: eval reward {avg:+.3f}')
    return adapter, history

## The experiment

In [ ]:
# Default sample: a learner-champion brain (hidden_size=14, attention head present)
# from the 30-min seed-1 run. Try 'final_overall_champion.json' or
# 'final_tool_champion.json' to compare different selection-pressure outputs.
SAMPLE_BRAIN = os.path.join(BRAIN_REPO, 'transfer/sample_brains/learner_champion_hidden14.json')

def random_baseline(env, n_episodes=200, seed=0):
    rng = np.random.default_rng(seed)
    rewards = []
    for _ in range(n_episodes):
        env.reset()
        total = 0.0
        done = False
        while not done:
            action = int(rng.integers(0, 3))
            _, r, done = env.step(action)
            total += r
        rewards.append(total)
    return float(np.mean(rewards))

env = Catch(rng=np.random.default_rng(7))

print('=== 1) Random policy ===')
random_score = random_baseline(env, n_episodes=200)
print(f'Random policy: {random_score:+.3f}\n')

print('=== 2) Untrained brain + trained adapter (control) ===')
random_brain = TinyBrainInference.random(input_size=72, hidden_size=14, output_size=15,
                                          rng=np.random.default_rng(7), with_attention=True)
control_adapter = CatchAdapter(random_brain, rng=np.random.default_rng(11))
control_adapter, control_history = train_es(control_adapter, env, generations=150, pop=32, seed=0)
control_score = evaluate(control_adapter, env, n_episodes=200)
print(f'Untrained brain + adapter: {control_score:+.3f}\n')

print('=== 3) Trained brain + trained adapter (transfer test) ===')
print(f'Loading brain from {SAMPLE_BRAIN}')
trained_brain = TinyBrainInference.from_checkpoint(SAMPLE_BRAIN)
print(f'  hidden={trained_brain.hidden_size}, input={trained_brain.input_size}, output={trained_brain.output_size}')
print(f'  attention head present: {trained_brain.attention_weights is not None}\n')
trained_adapter = CatchAdapter(trained_brain, rng=np.random.default_rng(11))
trained_adapter, trained_history = train_es(trained_adapter, env, generations=150, pop=32, seed=0)
trained_score = evaluate(trained_adapter, env, n_episodes=200)
print(f'Trained brain + adapter:   {trained_score:+.3f}')

## Results

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
if control_history:
    gens, scores = zip(*control_history)
    ax.plot(gens, scores, label='untrained brain + adapter (control)', marker='o')
if trained_history:
    gens, scores = zip(*trained_history)
    ax.plot(gens, scores, label='trained brain + adapter (transfer)', marker='o')
ax.axhline(random_score, color='gray', linestyle='--', label=f'random ({random_score:+.2f})')
ax.set_xlabel('Generation')
ax.set_ylabel('Mean reward (eval)')
ax.set_title('Microcosmic God brain → Catch transfer')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\nFinal scores (mean reward over 200 episodes, greedy policy):')
print(f'  Random policy:             {random_score:+.3f}')
print(f'  Untrained brain (control): {control_score:+.3f}')
print(f'  Trained brain (transfer):  {trained_score:+.3f}')
print(f'\n  Transfer advantage (trained − control): {trained_score - control_score:+.3f}')

## Skeptic's controls

The trained-brain advantage above is suggestive but not conclusive. Several alternative explanations could produce the same +0.38 advantage without the brain having actually learned anything transferable:

1. **No-brain direct policy** — Catch is solvable by `action = 1 + sign(ball_x − paddle_x)`, a 12-parameter linear policy. If a direct linear adapter (no brain at all) does as well as the brain-mediated version, the brain isn't contributing — the adapter is solving Catch despite the brain.
2. **Permutation control** — take the trained brain, randomly shuffle each weight matrix. This preserves the marginal weight distribution but destroys learned structure. If the shuffled brain transfers as well as the unshuffled one, the explanation is "trained brains have better-conditioned weight statistics" — not "trained brains learned useful representations."
3. **Multi-seed** — single-seed results can hide noise. Five seeds for each condition gives mean ± std and a more honest picture.

The decisive comparison is **trained vs permuted**. If they're statistically indistinguishable, the result is weight-statistics; if trained meaningfully exceeds permuted, learned structure is doing real work.

In [ ]:
def shuffle_brain(brain, rng):
    """Permute each weight matrix independently. Preserves marginal
    distribution of weights while destroying learned input-output and
    input-hidden structure."""
    def _shuf(arr):
        flat = arr.flatten().copy()
        rng.shuffle(flat)
        return flat.reshape(arr.shape)
    aw = _shuf(brain.attention_weights) if brain.attention_weights is not None else None
    ab = _shuf(brain.attention_bias) if brain.attention_bias is not None else None
    return TinyBrainInference(
        weights_in=_shuf(brain.weights_in),
        weights_out=_shuf(brain.weights_out),
        bias_h=_shuf(brain.bias_h),
        bias_o=_shuf(brain.bias_o),
        attention_weights=aw,
        attention_bias=ab,
    )


class DirectPolicy:
    """4 -> 3 linear policy with no brain at all. Drop-in replacement for
    CatchAdapter: same interface, ES can train it the same way. Tests
    whether the brain is helping or just being routed-around by the adapter."""

    def __init__(self, catch_obs_size=4, catch_action_size=3, rng=None):
        self.rng = rng or np.random.default_rng()
        self.W = self.rng.normal(0, 0.30, (catch_action_size, catch_obs_size))
        self.b = np.zeros(catch_action_size)

    def reset(self):
        pass

    def policy_logits(self, obs):
        return self.W @ obs + self.b

    def act_greedy(self, obs):
        return int(np.argmax(self.policy_logits(obs)))

    def act_stochastic(self, obs, temperature=1.0):
        l = self.policy_logits(obs) / max(1e-6, temperature)
        p = np.exp(l - l.max())
        p /= p.sum()
        return int(self.rng.choice(len(p), p=p))

    def get_params(self):
        return (self.W.copy(), self.b.copy())

    def set_params(self, W, b):
        self.W = W
        self.b = b

## Run all four conditions × 5 seeds

Five seeds × four conditions × 50 ES generations. Should take ~5-15 minutes on Colab CPU. Each condition uses fresh adapter init per seed; for the random-init brain and permuted-brain conditions, the brain itself is also re-sampled per seed.

In [ ]:
def run_seed(make_adapter, seed, gens=50, pop=24, n_avg=3, eval_eps=200):
    env = Catch(rng=np.random.default_rng(seed))
    adapter = make_adapter(seed)
    adapter, _ = train_es(
        adapter, env,
        generations=gens, pop=pop, n_avg=n_avg,
        seed=seed, log_every=gens, verbose=False,
    )
    return evaluate(adapter, env, n_episodes=eval_eps)


N_SEEDS = 5
SEEDS = list(range(20, 20 + N_SEEDS))

results = {}

print(f'Running {N_SEEDS} seeds per condition (this will take a few minutes) ...\n')

# Condition A: direct linear policy (no brain)
print('A) Direct linear policy (no brain) ...')
results['direct'] = [
    run_seed(lambda s: DirectPolicy(rng=np.random.default_rng(s)), s)
    for s in SEEDS
]
print('   ' + ' '.join(f'{x:+.2f}' for x in results['direct']))

# Condition B: random-init brain (different brain per seed)
print('B) Random-init brain + adapter (control) ...')
def random_brain_factory(s):
    rb = TinyBrainInference.random(input_size=72, hidden_size=14, output_size=15,
                                    rng=np.random.default_rng(s + 100), with_attention=True)
    return CatchAdapter(rb, rng=np.random.default_rng(s + 1000))
results['random_brain'] = [run_seed(random_brain_factory, s) for s in SEEDS]
print('   ' + ' '.join(f'{x:+.2f}' for x in results['random_brain']))

# Condition C: permuted trained brain (preserves weight distribution)
print('C) Permuted trained brain + adapter ...')
trained_brain = TinyBrainInference.from_checkpoint(SAMPLE_BRAIN)
def permuted_brain_factory(s):
    pb = shuffle_brain(trained_brain, rng=np.random.default_rng(s + 2000))
    return CatchAdapter(pb, rng=np.random.default_rng(s + 1000))
results['permuted'] = [run_seed(permuted_brain_factory, s) for s in SEEDS]
print('   ' + ' '.join(f'{x:+.2f}' for x in results['permuted']))

# Condition D: trained brain (the actual transfer test)
print('D) Trained brain + adapter (transfer test) ...')
def trained_brain_factory(s):
    return CatchAdapter(trained_brain, rng=np.random.default_rng(s + 1000))
results['trained'] = [run_seed(trained_brain_factory, s) for s in SEEDS]
print('   ' + ' '.join(f'{x:+.2f}' for x in results['trained']))

## The decisive comparison

Box plot across seeds, plus the three key contrasts.

In [ ]:
labels = ['Direct\n(no brain)', 'Random-init\nbrain', 'Permuted\ntrained brain', 'Trained\nbrain']
keys = ['direct', 'random_brain', 'permuted', 'trained']
data = [results[k] for k in keys]
positions = list(range(len(keys)))
colors = ['lightgray', 'lightblue', 'lightyellow', 'lightgreen']

fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot(data, positions=positions, widths=0.6, patch_artist=True)
for patch, c in zip(bp['boxes'], colors):
    patch.set_facecolor(c)
jitter_rng = np.random.default_rng(99)
for i, scores in enumerate(data):
    x = i + jitter_rng.uniform(-0.10, 0.10, size=len(scores))
    ax.scatter(x, scores, color='black', s=30, zorder=3)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xticks(positions)
ax.set_xticklabels(labels)
ax.set_ylabel('Mean reward (eval, 200 episodes per seed)')
ax.set_title(f'Catch transfer test - {N_SEEDS} seeds per condition')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nFinal scores (mean +/- std, n={N_SEEDS} seeds, 200 eval episodes each):')
for k, lbl in zip(keys, ['Direct (no brain)     ', 'Random-init brain     ', 'Permuted trained brain', 'Trained brain         ']):
    s = np.array(results[k])
    raw = [round(v, 2) for v in s.tolist()]
    print(f'  {lbl}: {s.mean():+.3f} +/- {s.std():.3f}   raw={raw}')

t_mean = np.mean(results['trained'])
delta_random = t_mean - np.mean(results['random_brain'])
delta_permuted = t_mean - np.mean(results['permuted'])
delta_direct = t_mean - np.mean(results['direct'])
print('\nKey contrasts:')
print(f'  trained - random_brain: {delta_random:+.3f}   (does training help over random?)')
print(f'  trained - permuted:     {delta_permuted:+.3f}   (is the *structure* doing the work, or just weight statistics?)')
print(f'  trained - direct:       {delta_direct:+.3f}   (does the brain help vs no brain at all?)')

## Interpreting the result

**Reward range:** Catch returns +1 on a catch, −1 on a miss. Per-episode reward is in {−1, +1}. A perfect player scores +1.0; uniform-random scores around 0.0.

**Reading the contrasts:**

| Contrast | What it tells you |
|---|---|
| `trained − permuted` | Whether *learned structure* (vs. just weight statistics) does work. If positive, shuffling the weights destroyed something useful. |
| `trained − random_brain` | Whether the alife training specifically produced something that transfers, beyond what any random brain would do. |
| `trained − direct` | Whether the brain helps at all vs. solving Catch with a 12-parameter linear policy. |

### What we observed (10 seeds, 150 ES generations, learner_champion_hidden14)

```
Direct (no brain)     : +0.084 ± 0.252
Random-init brain     : +0.004 ± 0.400
Permuted trained brain: -0.227 ± 0.197
Trained brain         : -0.020 ± 0.360

trained − permuted     = +0.207   ✓ structure matters
trained − random_brain = -0.024   ✗ no transfer beyond random
trained − direct       = -0.104   ✗ brain hinders slightly
```

**The permutation test passes** — shuffling weights makes the brain meaningfully worse. So the substrate IS producing structured representations, not just brains with reasonable weight statistics. That's a real positive finding.

**But the structure doesn't help with Catch specifically.** Trained and random brains transfer equally well; a direct linear policy slightly beats both. This is consistent with the read that Catch is too simple a test target — it's solvable by `action = 1 + sign(ball_x − paddle_x)`, a 12-parameter linear policy. A brain with rich representations of mg's world gets in the way of the adapter trying to extract that simple rule.

### What this means for the transferable-minds claim

The substrate is producing *structured cognition*, not just well-tuned random functions. That's necessary for transfer but isn't sufficient: structure has to be *relevant* to the target task.

A better future transfer target would be a task that genuinely requires:
- **temporal reasoning** (memory, prediction across multiple steps)
- **causal sequencing** (multi-step actions where order matters)
- **partial observability** (the brain's hidden state has to track unseen information)

The mg substrate was selected for exactly these. Catch tests none of them.

Candidates for a stronger second transfer test: a memory-based maze, a multi-step puzzle box (e.g., open-this-then-pull-that), or even a sequential prediction task where the brain's recurrent dynamics actually matter.

For now: the permutation result is the meaningful signal. The Catch result tells you Catch isn't the right probe.